In [ ]:
%pip install qiskit torch numpy pandas
%pip install scikit-learn

In [ ]:
import torch
import torch.nn as nn
from torch.autograd import Function
import numpy as np
import pandas as pd
from pathlib import Path # For OS-agnostic paths

from qiskit import QuantumCircuit
# from qiskit.circuit import ParameterVector # Not strictly needed for this direct value-binding approach
from qiskit.primitives import StatevectorEstimator # For Qiskit >= 0.46 (or qiskit.utils.QuantumInstance for older)
from qiskit.quantum_info import SparsePauliOp

from sklearn.model_selection import train_test_split # type: ignore
from sklearn.preprocessing import StandardScaler # type: ignore

from typing import Tuple, List, Any, Optional

# --- 1. Data Loading and Preprocessing ---
# Ensure the data file is in a 'data' subdirectory or adjust the path.
# Original code used: "data\\wdbc.data"
data_file_path = Path("data") / "wdbc.data"
# Fallback if the above path doesn't exist (e.g., running in a different CWD)
if not data_file_path.exists():
    data_file_path = Path("wdbc.data") # Check current directory

try:
    df = pd.read_csv(data_file_path, header=None)
except FileNotFoundError:
    print(f"Error: Could not find the data file at '{data_file_path.resolve()}' or 'wdbc.data'.")
    print("Please ensure 'wdbc.data' is in a 'data' subdirectory relative to your script, or in the current directory.")
    exit()


# Assign column names
columns = ['id', 'diagnosis'] + [f'feature_{i}' for i in range(1, 31)]
df.columns = columns

# Drop the ID column
df = df.drop(columns=['id'])

# Encode diagnosis: M = 1, B = 0
df['diagnosis'] = df['diagnosis'].map({'M': 1, 'B': 0})

# Separate features and target
X_numpy = df.drop(columns=['diagnosis']).values.astype(np.float32)
Y_numpy = df['diagnosis'].values.astype(np.float32).reshape(-1, 1)

# Normalize features using sklearn
scaler = StandardScaler()
X_numpy_scaled = scaler.fit_transform(X_numpy)

# Convert to torch tensors
X_tensor_all = torch.tensor(X_numpy_scaled, dtype=torch.float32)
Y_tensor_all = torch.tensor(Y_numpy, dtype=torch.float32)

# Train/test split using sklearn
X_train, X_test, Y_train, Y_test = train_test_split(
    X_tensor_all, Y_tensor_all, test_size=0.2, random_state=42, stratify=Y_tensor_all
)

print(f"Training set size: X_train: {X_train.shape}, Y_train: {Y_train.shape}")
print(f"Test set size: X_test: {X_test.shape}, Y_test: {Y_test.shape}")

# --- 2. VQA Circuit Definition ---
N_QUBITS = 2  # Number of qubits, matches the output of the first classical layer (2 features)
N_QUANTUM_WEIGHTS = 4 # Number of trainable parameters for the VQA circuit, as in original self.weights

def create_vqa_circuit(input_data: np.ndarray, weights: np.ndarray) -> QuantumCircuit:
    """
    Creates a VQA circuit with encoded input data and trainable weights.
    This function replaces the placeholder in your original `create_vqa_circuit`.

    Args:
        input_data: A 1D numpy array with 2 elements (features from classical layer).
        weights: A 1D numpy array with N_QUANTUM_WEIGHTS elements (trainable quantum parameters).

    Returns:
        A Qiskit QuantumCircuit.
    """
    if not isinstance(input_data, np.ndarray) or input_data.ndim != 1 or input_data.shape[0] != 2:
        raise ValueError(f"Expected input_data to be a 1D numpy array with 2 elements, got {input_data}")
    if not isinstance(weights, np.ndarray) or weights.ndim != 1 or weights.shape[0] != N_QUANTUM_WEIGHTS:
        raise ValueError(f"Expected weights to be a 1D numpy array with {N_QUANTUM_WEIGHTS} elements, got {weights}")

    qc = QuantumCircuit(N_QUBITS)

    # Encode input data (e.g., using RY rotations)
    # These two values come from the preceding classical layer
    qc.ry(input_data[0], 0)
    qc.ry(input_data[1], 1)
    qc.barrier()

    # Variational layer (trainable weights)
    # Using 4 weights as defined by nn.Parameter(torch.randn(4)) in original VQALayer
    qc.rz(weights[0], 0)
    qc.ry(weights[1], 0)
    qc.rz(weights[2], 1)
    qc.ry(weights[3], 1)

    # Entanglement
    qc.cx(0, 1)
    # qc.barrier() # Optional: add more barriers/layers if desired

    return qc

# --- 3. Qiskit Estimator and Observables ---
# Using StatevectorEstimator for exact expectation values (suitable for small circuits)
# It's recommended to instantiate this once if its configuration doesn't change.
estimator = StatevectorEstimator() # For Qiskit < 1.0, you might use Estimator() from qiskit.primitives
# Observable: Measure Pauli Z on the first qubit, Identity on the others (here, on the second qubit)
observables = [SparsePauliOp("ZI")] # ZI means Z on qubit 0, I on qubit 1.

# --- 4. PyTorch Custom Autograd Function For VQA Layer ---
class VQALayerFunction(Function):
    @staticmethod
    def forward(ctx: Any, input_tensor_single: torch.Tensor, weights: torch.Tensor) -> torch.Tensor:
        """
        Forward pass for a single data sample.
        input_tensor_single: Tensor for one sample, shape (2,) after classical layer.
        weights: Quantum circuit parameters, shape (N_QUANTUM_WEIGHTS,).
        """
        # Ensure data is on CPU and in NumPy format for Qiskit
        input_vals = input_tensor_single.detach().cpu().numpy()
        weight_vals = weights.detach().cpu().numpy()

        # Save tensors for backward pass
        ctx.save_for_backward(input_tensor_single, weights) # Save original torch tensors

        # Create and run the quantum circuit for the single input sample
        qc = create_vqa_circuit(input_vals, weight_vals)

        # Estimator runs a list of PUBs (Primitive Unified Blocs)
        # Each PUB is (circuit, observable, parameter_values (optional if bound))
        pub = (qc, observables[0]) # circuit is already bound with numerical values
        job = estimator.run([pub]) # Pass as a list of one PUB
        result = job.result()

        # Extract expectation value
        # result[0] corresponds to the first PUB, .data.evs[0] for the first (and only) observable's value
        expval_scalar = result[0].data.evs.item()

        # Return as a tensor on the same device as input_tensor_single
        return torch.tensor([expval_scalar], dtype=torch.float32, device=input_tensor_single.device)

    @staticmethod
    def backward(ctx: Any, grad_output: torch.Tensor) -> Tuple[Optional[torch.Tensor], Optional[torch.Tensor]]:
        """
        Backward pass using the parameter-shift rule.
        This function replaces the placeholder in your original VQALayerFunction.backward.
        grad_output: Gradient from the next layer (scalar for a single sample's output).
        """
        input_tensor_single, weights = ctx.saved_tensors # Retrieve saved torch tensors
        input_vals = input_tensor_single.detach().cpu().numpy()
        weight_vals = weights.detach().cpu().numpy()

        shift = np.pi / 2  # Parameter shift amount
        grads_np = np.zeros_like(weight_vals, dtype=np.float32)

        # Prepare list of circuits for batched execution (parameter-shift rule)
        # This is the core of the speed improvement for the gradient calculation.
        pubs_for_gradient_calc: List[Tuple[QuantumCircuit, SparsePauliOp]] = []

        for i in range(len(weight_vals)): # Iterate over each trainable quantum weight
            # Shift positive
            weights_plus = weight_vals.copy()
            weights_plus[i] += shift
            qc_plus = create_vqa_circuit(input_vals, weights_plus)
            pubs_for_gradient_calc.append((qc_plus, observables[0]))

            # Shift negative
            weights_minus = weight_vals.copy()
            weights_minus[i] -= shift
            qc_minus = create_vqa_circuit(input_vals, weights_minus)
            pubs_for_gradient_calc.append((qc_minus, observables[0]))

        # Run all circuits for gradient calculation in one batch if list is not empty
        if pubs_for_gradient_calc:
            job = estimator.run(pubs_for_gradient_calc)
            results_gradient = job.result()

            for i in range(len(weight_vals)):
                # Result for qc_plus for i-th weight
                expval_plus = results_gradient[2 * i].data.evs.item()
                expval_minus = results_gradient[2 * i + 1].data.evs.item()
                # Parameter-shift rule: (f(x+s) - f(x-s)) / (2*sin(s))
                # For s=pi/2 and Pauli observable (eigenvalues +/-1), simplifies to (E_plus - E_minus) / 2
                grads_np[i] = (expval_plus - expval_minus) / 2.0

        # Convert gradients to tensor and scale by grad_output from the next layer
        # grad_output is a scalar here, it will broadcast correctly.
        final_grads_for_weights = torch.tensor(grads_np, dtype=torch.float32, device=weights.device) * grad_output

        # Gradients are for 'weights'. No gradient for 'input_tensor_single' in this common VQA setup,
        # as input_tensor_single is treated as data encoding parameters, not trainable parameters of this layer.
        return None, final_grads_for_weights


# --- 5. Quantum Layer as PyTorch Module ---
class VQALayer(nn.Module):
    def __init__(self, num_quantum_weights: int = N_QUANTUM_WEIGHTS):
        super().__init__()
        # Initialize weights for the quantum circuit
        # Original code: self.weights = nn.Parameter(torch.randn(4))
        self.quantum_weights = nn.Parameter(torch.randn(num_quantum_weights))
        # Can also use other initializations, e.g., uniform:
        # nn.init.uniform_(self.quantum_weights, a=-np.pi, b=np.pi)


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Apply VQALayerFunction to each sample in the batch.
        x: Batch of inputs from classical layer, shape (batch_size, num_input_features_to_quantum_layer)
           Here, num_input_features_to_quantum_layer should be 2.
        """
        # Process each sample in the batch individually by applying the custom autograd Function.
        # This is typical for VQAs where each input modifies the circuit parameters
        # or effectively requires a separate circuit execution.
        # The VQALayerFunction itself handles the quantum computation for a single sample.
        results_list = [VQALayerFunction.apply(x[i], self.quantum_weights) for i in range(x.size(0))]

        # Stack results from all samples in the batch. Each result is a 1-element tensor.
        # The .view(-1, 1) ensures the output shape is (batch_size, 1).
        return torch.stack(results_list).view(-1, 1)

# --- 6. Full Hybrid Model ---
class HybridModel(nn.Module):
    def __init__(self, num_original_input_features: int,
                 num_classical_hidden_features: int = 2, # Output of classical layer, input to quantum
                 num_quantum_weights: int = N_QUANTUM_WEIGHTS):
        super().__init__()
        # Classical pre-processing layer: maps original input features to `num_classical_hidden_features`
        # Original code: self.classical = nn.Linear(X_tensor.shape[1], 2)
        self.classical_preprocess = nn.Linear(num_original_input_features, num_classical_hidden_features)
        # Quantum layer
        self.quantum_layer = VQALayer(num_quantum_weights)
        # Final classical layer to map quantum output (1 expectation value) to 1 logit for classification
        self.output_layer = nn.Linear(1, 1) # Takes 1 output from quantum layer

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.classical_preprocess(x)
        x = torch.tanh(x)  # Activation before quantum layer (common choice)
        x = self.quantum_layer(x)
        x = self.output_layer(x) # Output layer (logit)
        return torch.sigmoid(x) # Sigmoid for binary classification probability

# --- 7. Model Initialization, Optimizer, and Loss ---
# Determine input features from the training data
num_features_from_data = X_train.shape[1]

model = HybridModel(
    num_original_input_features=num_features_from_data,
    num_classical_hidden_features=2, # This must match what create_vqa_circuit expects for input_data
    num_quantum_weights=N_QUANTUM_WEIGHTS
)

optimizer = torch.optim.Adam(model.parameters(), lr=0.05) # lr can be tuned
loss_fn = nn.BCELoss() # Binary Cross-Entropy Loss for binary classification

# --- 8. Training Loop ---
epochs = 25 # Original was 50; adjust as needed
batch_size = 16 # Introduce batching for more stable training

print(f"\nStarting Training for {epochs} epochs with batch size {batch_size}...")
for epoch in range(epochs):
    model.train() # Set model to training mode
    permutation = torch.randperm(X_train.size(0)) # Shuffle data at the start of each epoch
    
    running_loss = 0.0
    correct_predictions_train = 0
    total_train_samples = 0

    for i in range(0, X_train.size(0), batch_size):
        optimizer.zero_grad() # Zero gradients for each batch

        indices = permutation[i : i + batch_size]
        batch_X, batch_Y = X_train[indices], Y_train[indices]

        preds = model(batch_X)
        loss = loss_fn(preds, batch_Y)
        
        loss.backward() # Compute gradients
        optimizer.step() # Update weights

        running_loss += loss.item() * batch_X.size(0)
        predicted_labels = (preds > 0.5).float()
        correct_predictions_train += (predicted_labels == batch_Y).sum().item()
        total_train_samples += batch_X.size(0)

    epoch_loss = running_loss / total_train_samples
    epoch_acc_train = correct_predictions_train / total_train_samples * 100

    # Validation phase (optional, but good practice)
    model.eval() # Set model to evaluation mode
    with torch.no_grad(): # Disable gradient calculations for validation
        val_preds = model(X_test)
        val_loss = loss_fn(val_preds, Y_test).item()
        val_predicted_labels = (val_preds > 0.5).float()
        val_acc = (val_predicted_labels == Y_test).float().mean().item() * 100
    
    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc_train:.2f}% | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

print("\nTraining Finished.")

# Example of final evaluation on test set
model.eval()
with torch.no_grad():
    final_test_preds = model(X_test)
    final_test_loss = loss_fn(final_test_preds, Y_test)
    final_test_acc = ((final_test_preds > 0.5).float() == Y_test).float().mean()
    print(f"\nFinal Test Results: Loss: {final_test_loss.item():.4f} | Accuracy: {final_test_acc.item()*100:.2f}%")